# Prolog in SCRIP: the core rungs

### asserta_assertz_directive_1

In [ ]:
%%prolog
:- initialization(main).
:- assertz(item(b)).
:- assertz(item(c)).
:- asserta(item(a)).

main :-
    item(X),
    write(X), nl,
    fail.
main.

### clause_source_record__csr_listing_names

In [ ]:
%%prolog
:- use_module(library(lists)).
:- initialization(main).
:- op(700, xfx, rplus).
:- dynamic(dummy/4).

goal_expansion(L rplus R, R + L).

dummy(A,B,C,D) :- B rplus A, D rplus C.

dummy(A,B) --> { B rplus A }.

dummy((Head --> Goals), Clause, _, _) :-
        list_goal(Goals, Body),
        expand_term((Head --> Body), Clause).

dummy(g(Goal), []) --> [{Goal}, !].

dummy(Foo, _Bar, _, Baz) :-
        Baz = [Foo|_].

main :- listing(dummy/4).

### asserta_order_1

In [ ]:
%%prolog
:- dynamic(g/1).
:- initialization(main).
main :- assertz(g(1)), assertz(g(2)), asserta(g(0)), findall(X, g(X), L), write(L), nl.

### op_ite_directive_replace_2

In [ ]:
%%prolog
:- op(200, fy, ~).
:- initialization(main).
main :-
    ( ~ ~ a == ~(~(a)) -> write(yes) ; write(no) ), nl,
    ( X = ~ foo, X == ~(foo) -> write(yes) ; write(no) ), nl,
    ( ~ (b + c) == ~(b + c) -> write(yes) ; write(no) ), nl.

### call_directive_replace_3

In [ ]:
%%prolog
% rung33_bridge_callN/05_call2_compound_g — call/2 where G is already a compound.
% call(G, ExtraArg) where G=succ(3) reconstructs succ(3, R) — G's args
% prepended, extra args appended. Tests the compound-G case of call/N.
:- initialization(main).
main :-
    G = succ(3),
    call(G, R),
    write(R), nl.

### op_ite_format_1

In [ ]:
%%prolog
:- initialization(main).
main :-
    op(600, xfx, rt_infix),
    op(300, fy, rt_pre),
    ( current_op(P1, T1, rt_infix) -> format("~w ~w~n", [P1, T1]) ; write(none), nl ),
    ( current_op(P2, T2, rt_pre) -> format("~w ~w~n", [P2, T2]) ; write(none), nl ),
    op(400, yfx, rt_multi),
    ( current_op(400, yfx, rt_multi) -> write(yes) ; write(no) ), nl.

### dcg_ite_list_replace_1

In [ ]:
%%prolog
:- initialization(main).

digits([D|Ds]) --> digit(D), digits(Ds).
digits([]) --> [].
digit(D) --> [D], { D >= 0'0, D =< 0'9 }.

main :-
    atom_codes('123', Codes),
    ( phrase(digits(Ds), Codes) ->
        atom_codes(A, Ds), write(A)
    ; write(fail)
    ), nl.

### ladder__rung10_monotonic_tabling_propagates_an_assert_into_its_dependents

In [ ]:
%%prolog
:- dynamic da/1 as monotonic.
:- table pa/1 as monotonic.
pa(X) :- da(Y), X is Y+1.
:- dynamic dx/1 as (incremental,monotonic).
:- table px/1 as monotonic.
:- table qx/1 as incremental.
px(X) :- dx(Y), X is Y+1.
qx(X) :- px(Y), X is Y*2.
:- table cm/2 as monotonic.
:- dynamic lm/2 as monotonic.
cm(X, Y) :- cm(Y, X).
cm(X, Z) :- cm(X, Y), cm(Y, Z).
cm(X, Y) :- lm(X, Y).
main :-
    ( pa(_) -> writeln(pa_nonempty) ; writeln(pa_empty) ),
    assert(da(0)),
    ( get_returns_for_call(pa(_), pa(1)) -> writeln(ret_pa1) ; writeln(no_ret_pa1) ),
    findall(X, pa(X), L1), writeq(L1), nl,
    ( incr_is_invalid(pa(_)) -> writeln(pa_invalid) ; writeln(pa_valid) ),
    assert(dx(1)), qx(Q1), writeq(q(Q1)), nl,
    assert(dx(2)),
    ( incr_is_invalid(qx(_)) -> writeln(qx_invalid) ; writeln(qx_valid) ),
    findall(Q, qx(Q), QL), sort(QL, QS), writeq(QS), nl,
    ignore(cm(_,_)),
    assert(lm(a, b)), assert(lm(a, c)),
    findall(X-Y, cm(X,Y), Pairs), length(Pairs, Len), writeq(len(Len)), nl,
    sort(Pairs, SP), writeq(SP), nl.
:- initialization(main).

### typetest_directive_1

In [ ]:
%%prolog
:- initialization(main).
main :- a == a, a \== b, a @< b, b @> a, a @=< a, b @>= a, compare(O, 1, 2), atom(foo), number(3), integer(3), float(1.5), atomic(foo), var(_), nonvar(foo), compound(f(x)), callable(foo), write(O), nl, write(ok), nl.

### abolish_abolish_removes_clauses_1

In [ ]:
%%prolog
:- dynamic(aa10a/1).
:- initialization(main).
main :- assertz(aa10a(1)), assertz(aa10a(2)), abolish(aa10a/1), catch((findall(X,aa10a(X),L), write(L)), _, write(gone)), nl.

### assertz_clause_call_1

In [ ]:
%%prolog
:- initialization(main).
main :- assertz((greet :- write(hello), nl)),
        ( clause(greet, Body) -> call(Body) ; write(noclause), nl ).

### assertz_retract_directive_2

In [ ]:
%%prolog
:- dynamic(f/1).
:- initialization(main).
main :- assertz(f(1)), assertz(f(2)), retract(f(1)), findall(X, f(X), L), write(L), nl.

### retract_backtrack_1

In [ ]:
%%prolog
:- dynamic(h/1).
:- initialization(main).
main :- assertz(h(1)), assertz(h(2)), assertz(h(3)), (retract(h(_)), fail ; true), findall(X, h(X), L), write(L), nl.